In [12]:
!pip install transformers datasets accelerate -q

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

In [14]:
data = {
    "text": [
        "I feel stressed and anxious. I understand, take deep breaths and stay calm.",
        "I am feeling sad today. It's okay to feel sad, you are not alone.",
        "I am very worried about my future. Try to focus on small steps each day.",
        "I feel lonely. You are important and things will get better."
    ]
}

dataset = Dataset.from_dict(data)

In [15]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_data = dataset.map(tokenize)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [17]:
training_args = TrainingArguments(
    output_dir="./task5_model",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=1,
    report_to="none"
)

In [18]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    data_collator=data_collator
)

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,4.063011
2,3.206605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2, training_loss=3.634808301925659, metrics={'train_runtime': 32.5948, 'train_samples_per_second': 0.123, 'train_steps_per_second': 0.061, 'total_flos': 261292032000.0, 'train_loss': 3.634808301925659, 'epoch': 1.0})

In [22]:
def chatbot(text):
    input_text = f"User: {text} Assistant:"
    inputs = tokenizer.encode(input_text, return_tensors="pt")

    output = model.generate(inputs, max_length=120, do_sample=True)

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [23]:
print(chatbot("I feel very stressed"))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


User: I feel very stressed Assistant: I feel very stressed and depressed my head is doing extremely well

It's hard to watch TV: It's easy to watch TV: 1) you are in control 2) You are in control 5) The truth is about everyone else does not know your truth

You work together: It hurts me to work with each other more than time to do them: it may take you

You make good friends: You make good friends and they do good, you make good friends

You go to the gym but keep good: You go to
